# Lunar Simulation v001

Joint sky+beam recovery for the lunar campaign using `Calibrator`.
Sections 1–7 are unchanged from v000 (campaign setup, orbital mechanics,
tumble dynamics, beam and geometry visualisation).  Sections 8–12 replace
the v000 linear design-matrix solver with the Anderson-accelerated
Newton-CG `Calibrator`, run in two modes:

- **Single spacecraft** (section 10): spacecraft-0 data only — establishes a baseline.
- **Stacked** (section 11): both spacecraft concatenated through a `StackedForwardModel` wrapper — uses the full sky coverage and doubles the data volume.

## 1. Configuration And Frames

Galactic coordinates are the inertial sky frame.  Orbit normals are defined
in the J2000 mean ecliptic frame, with shared vernal-equinox ascending node,
then transformed to Galactic coordinates.  Body rotations map inertial vectors
into the crossed-dipole frame.

In [ ]:
import time
import numpy as np
import healpy
import jax.numpy as jnp
import matplotlib.pyplot as plt
from astropy.time import Time
import astropy.units as u
from eigsep_sim import (
    LunarCampaign, LunarRecoveryAdapter, Calibrator,
)
from eigsep_sim.models import T21cmModel
from eigsep_sim.spectral import gsm_eigenmodes, eigenmode_filter
from eigsep_sim.lunar import angular_momentum_for_spin_period, crossed_rod_inertia, integrate_torque_free

FIGSIZE = (12, 4)

profile = "proposal"  # single switch: use "proposal" for the larger study
profiles = {
    "interactive": {"nside": 16, "nchan": 32, "ntimes": 64, "hours": 4.0},
    "proposal":    {"nside": 32, "nchan": 32, "ntimes": 128, "hours": 4.0},
}
p = profiles[profile]
config = {
    "profile": profile,
    "spacecraft": {
        "opening_angle_deg": 90.0, "arm_lengths_m": [6.0, 4.0],
        "arm_masses_kg": [1.0, 2.25], "angular_momentum_direction_gal": [1.0, 0.0, 0.2],
        "spin_period_s": 60.0,
        "attitude_phase_offsets_deg": [0.0, 45.0],
    },
    "orbit": {
        "altitude_km": 100.0, "inclinations_deg": [-15.0, 15.0],
        "ascending_node_lon_deg": 0.0, "equinox": "J2000", "epoch": "2025-01-01",
    },
    "antenna": {"nside": p["nside"], "n_modes": 3},
    "receiver": {"T_rx_K": 100.0},
    "sky": {"nside": p["nside"], "n_modes": min(3, p["nchan"] - 1)},
    "frequency": {"min_mhz": 55.0, "max_mhz": 145.0, "nchan": p["nchan"]},
    "integration": {"duration_hours": p["hours"], "ntimes": p["ntimes"], "attitude_step_s": 10.0},
    "recovery": {"include_receiver_offsets": False, "n_eig_modes": 3},
    "monte_carlo": {"nreal": 8 if profile == "interactive" else 200, "seed": 0},
    "surface": {"T_regolith_K": 300.0, "reflectivity_enabled": False},
    "signal_21cm": {"enabled": True, "model_index": 0},
    "sources": {"earth": {"enabled": False}, "sun": {"enabled": False}},
}
config

## 2. Moon, Orbits, And Galactic Coverage

In [ ]:
campaign = LunarCampaign(config)
result = campaign.run()
freqs_mhz = campaign.freqs_hz / 1e6

fig = plt.figure(figsize=FIGSIZE)
ax = fig.add_subplot(121, projection="3d")
u_arc = np.linspace(0, 2*np.pi, 80)
for normal, color in zip(campaign.orbit_normals_gal, ["C0", "C1"]):
    ref = np.cross(normal, [0, 0, 1])
    if np.linalg.norm(ref) < 1e-6: ref = np.cross(normal, [0, 1, 0])
    ref /= np.linalg.norm(ref); ortho = np.cross(normal, ref)
    xyz = np.cos(u_arc)[:, None]*ref + np.sin(u_arc)[:, None]*ortho
    ax.plot(*xyz.T, color=color)
ax.scatter([0], [0], [0], s=180, color="0.6"); ax.set_title("Moon and circular orbit planes")
plt.subplot(122, projection="mollweide")
visible = result.masks.any(axis=(0, 1)); theta, phi = healpy.pix2ang(campaign.sky.nside, np.where(visible)[0])
plt.scatter(np.pi-phi, np.pi/2-theta, s=8); plt.title("Ever-visible Galactic sky pixels")
plt.tight_layout()

## 3. Torque-Free Tumble Invariants And Arm Coverage

In [ ]:
I = crossed_rod_inertia(config["spacecraft"]["arm_lengths_m"], config["spacecraft"]["arm_masses_kg"])
t = np.linspace(0, 60 * config['integration']['duration_hours'], config['integration']['ntimes'])
L = angular_momentum_for_spin_period(I, config["spacecraft"]["angular_momentum_direction_gal"], config["spacecraft"]["spin_period_s"])
tumble = integrate_torque_free(I, L, t)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(t, tumble["kinetic_energy"] / tumble["kinetic_energy"][0] - 1); ax[0].set_title("Fractional kinetic-energy drift")
axes = tumble["rotations_body_to_gal"].apply(np.broadcast_to(campaign.arm_axes_body[0], (len(t), 3)))
ax[1].scatter(np.arctan2(axes[:,1], axes[:,0]), np.arcsin(axes[:,2]), s=3); ax[1].set_title("Arm-axis pointing coverage")
plt.tight_layout()

## 4. GSM Maps And Injected 21-cm Ensemble

In [ ]:
gsm_plus_signal = campaign.sky.basis.deproject(campaign.sky_coeffs)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(freqs_mhz, gsm_plus_signal.mean(axis=0), label="GSM + T21")
ax[0].plot(freqs_mhz, campaign.T21cm_K, label="injected T21"); ax[0].legend()
models = T21cmModel()(campaign.freqs_hz)
modes = gsm_eigenmodes(gsm_plus_signal, min(config["recovery"]["n_eig_modes"], len(freqs_mhz)-1))
filtered_models = eigenmode_filter(models, modes)
filtered_injected = eigenmode_filter(campaign.T21cm_K, modes)
ax[1].plot(freqs_mhz, filtered_models.T, alpha=.15); ax[1].set_title("Eigenmode-filtered signal ensemble")
plt.tight_layout()

## 5. BODY-Frame Dipole Beams

In [ ]:
beam_maps = campaign.beam.basis.deproject(campaign.beam.coeffs)
fig, ax = plt.subplots(1, 2, figsize=FIGSIZE)
for d in range(2):
    healpy.mollview(beam_maps[d, :, len(freqs_mhz)//2], fig=fig.number, sub=(1,2,d+1), title=f"Dipole {d} BODY beam")
plt.tight_layout()

## 6. GAL-Frame Masks, Disk Emission, Beam Sampling, And Weights

In [ ]:
adapter = LunarRecoveryAdapter(campaign, result)
weights = adapter.beam_weights(0, len(freqs_mhz)//2)
fig, ax = plt.subplots(1, 2, figsize=FIGSIZE)
healpy.mollview(result.masks[0,0], fig=fig.number, sub=(1,2,1), title="GAL sky mask: spacecraft 0")
healpy.mollview(weights[0,0], fig=fig.number, sub=(1,2,2), title="Normalized integration weights")
plt.tight_layout()

## 7. Full Versus Reduced JAX Geometry

In [ ]:
fwd0 = campaign.forward_models[0]
fwd1 = campaign.forward_models[1]
times = result.times
body_rots = result.body_rots[0]

t0 = time.perf_counter()
full_geom = fwd0.precompute_geometry(times=times, body_rots=body_rots)
full_dt = time.perf_counter() - t0

sky_mask = fwd0.build_sky_mask(times=times)
t0 = time.perf_counter()
reduced_geom = fwd0.precompute_geometry(times=times, body_rots=body_rots, sky_mask=sky_mask)
reduced_dt = time.perf_counter() - t0

T_full = np.asarray(fwd0.simulate(campaign.sky_coeffs, campaign.beam.coeffs, geom=full_geom))
T_reduced = np.asarray(fwd0.simulate(campaign.sky_coeffs, campaign.beam.coeffs, geom=reduced_geom))
print({"max_abs_K": float(np.max(np.abs(T_full - T_reduced))),
       "full_s": full_dt, "reduced_s": reduced_dt,
       "pixels_kept": int(sky_mask.sum())})

## 8. Noisy Observations

Radiometer noise following $\sigma_\nu = T_{\rm sys}/\sqrt{\Delta\nu\,\tau}$
where $T_{\rm sys} = \langle T_{\rm sky}\rangle + T_{\rm rx}$.
Noise is the same for both spacecraft (shared beam, same orbit altitude).

In [ ]:
delta_nu_hz = float(np.diff(campaign.freqs_hz).mean())
tau_s = config["integration"]["duration_hours"] * 3600.0 / config["integration"]["ntimes"]
T_rx_K = float(config["receiver"]["T_rx_K"])

sky_mean_K = gsm_plus_signal.mean(axis=0)  # (nfreq,)
sigma_noise = (sky_mean_K + T_rx_K) / np.sqrt(delta_nu_hz * tau_s)  # (nfreq,)

rng = np.random.default_rng(seed=42)

# Per-spacecraft noisy data: shape (ntimes, n_dipoles, nfreq)
truth_sc0 = result.spectra_K[0].astype(np.float32)
truth_sc1 = result.spectra_K[1].astype(np.float32)
data_sc0 = truth_sc0 + rng.normal(scale=sigma_noise[None, None, :], size=truth_sc0.shape).astype(np.float32)
data_sc1 = truth_sc1 + rng.normal(scale=sigma_noise[None, None, :], size=truth_sc1.shape).astype(np.float32)

print(f"Per-spacecraft data shape: {data_sc0.shape}  (ntimes, n_dipoles, nfreq)")
print(f"sigma_noise: {sigma_noise.mean()*1e3:.1f} mK (mean across freq)")

## 9. Stacked Forward Model

`StackedForwardModel` wraps two `ForwardModel` instances that share the same
sky and beam.  Its `simulate` method calls each model with its own geometry
and concatenates the outputs along the time axis.  Because both legs use
`jax.lax.scan` internally, gradients flow through the concatenation back to
`sky_coeffs` and `beam_coeffs` correctly.

In [ ]:
class StackedForwardModel:
    """Two ForwardModels with shared sky/beam; outputs concatenated along time.

    Parameters
    ----------
    fwd_list : list of ForwardModel
        Each entry must share the same sky and beam objects.

    Notes
    -----
    ``simulate`` accepts ``geom`` as a **list** of geometry dicts, one per
    forward model, matching the order of ``fwd_list``.
    """

    def __init__(self, fwd_list):
        self.fwd_list = list(fwd_list)
        # Expose sky/beam for Calibrator.init_params
        self.sky = fwd_list[0].sky
        self.beam = fwd_list[0].beam

    def simulate(self, sky_coeffs, beam_coeffs, geom=None, **kwargs):
        """Simulate and concatenate across all spacecraft.

        Parameters
        ----------
        sky_coeffs, beam_coeffs
            Shared coefficient arrays (JAX-traced inside Calibrator).
        geom : list of dict
            Pre-computed geometry dicts, one per entry in fwd_list.

        Returns
        -------
        jnp.ndarray, shape (n_sc * ntimes, n_dipoles, nfreq)
        """
        outputs = [
            fwd.simulate(sky_coeffs, beam_coeffs, geom=g, **kwargs)
            for fwd, g in zip(self.fwd_list, geom)
        ]
        return jnp.concatenate(outputs, axis=0)


stacked_fwd = StackedForwardModel(campaign.forward_models)
stacked_geom = list(result.geometry)           # [geom_sc0, geom_sc1]
data_stacked = np.concatenate([data_sc0, data_sc1], axis=0)  # (2*ntimes, n_dip, nfreq)
inv_noise_var_stacked = np.broadcast_to(
    1.0 / sigma_noise[None, None, :]**2, data_stacked.shape
).copy().astype(np.float32)

print(f"Stacked data shape: {data_stacked.shape}")
print(f"Verify stacked simulation shape: {stacked_fwd.simulate(campaign.sky_coeffs, campaign.beam.coeffs, geom=stacked_geom).shape}")

## 10. Joint Calibration — Spacecraft 0

Baseline: recover sky and beam from spacecraft-0 data only.

In [ ]:
inv_noise_var_sc0 = np.broadcast_to(
    1.0 / sigma_noise[None, None, :]**2, data_sc0.shape
).copy().astype(np.float32)

cal_single = Calibrator(
    fwd=fwd0,
    data=data_sc0,
    inv_noise_var=inv_noise_var_sc0,
    m_anderson=5,
    lam_beam=0.01,
    lam_sky=0.0,
)
params_single = cal_single.init_params(geom=result.geometry[0])
params_single["sky_coeffs"]  = campaign.sky_coeffs * 1.2
params_single["beam_coeffs"] = campaign.beam.coeffs * 0.9

print("Running single-spacecraft fit …", flush=True)
fit_single = cal_single.fit(
    params=params_single, geom=result.geometry[0],
    max_iter=20, tol=1e-4, verbose=True, use_joint=True,
)
print(f"\n✓ Single-spacecraft: converged={fit_single['converged']} in {fit_single['n_iter']} iter")

## 11. Joint Calibration — Both Spacecraft Stacked

The `StackedForwardModel` combines data and geometry from both spacecraft.
The sky and beam are shared, so the optimiser sees twice the data volume
and twice the sky coverage in each gradient step.

In [ ]:
cal_stacked = Calibrator(
    fwd=stacked_fwd,
    data=data_stacked,
    inv_noise_var=inv_noise_var_stacked,
    m_anderson=5,
    lam_beam=0.01,
    lam_sky=0.0,
)
params_stacked = cal_stacked.init_params(geom=stacked_geom)
params_stacked["sky_coeffs"]  = campaign.sky_coeffs * 1.2
params_stacked["beam_coeffs"] = campaign.beam.coeffs * 0.9

print("Running stacked-spacecraft fit …", flush=True)
fit_stacked = cal_stacked.fit(
    params=params_stacked, geom=stacked_geom,
    max_iter=20, tol=1e-4, verbose=True, use_joint=True,
)
print(f"\n✓ Stacked: converged={fit_stacked['converged']} in {fit_stacked['n_iter']} iter")

## 12. Recovery Comparison

Convergence, sky map residuals, and 21-cm eigenmode SNR for single vs stacked.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(fit_single["losses"],  "C0o-", markersize=5, label="spacecraft 0")
ax.semilogy(fit_stacked["losses"], "C1s-", markersize=5, label="stacked (both)")
ax.set_xlabel("Iteration"); ax.set_ylabel("Loss")
ax.set_title("Calibrator Convergence")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()

In [ ]:
fi = len(freqs_mhz) // 2  # mid-band channel
sky_truth = campaign.sky.basis.deproject(campaign.sky_coeffs)  # (npix, nfreq)

sky_single  = campaign.sky.basis.deproject(fit_single["params"]["sky_coeffs"])
sky_stacked = campaign.sky.basis.deproject(fit_stacked["params"]["sky_coeffs"])

resid_single  = (sky_single[:, fi]  - sky_truth[:, fi]) / sky_truth[:, fi]
resid_stacked = (sky_stacked[:, fi] - sky_truth[:, fi]) / sky_truth[:, fi]

vmax = max(np.nanpercentile(np.abs(resid_single), 99),
           np.nanpercentile(np.abs(resid_stacked), 99))

fig = plt.figure(figsize=(14, 5))
healpy.mollview(sky_truth[:, fi],    fig=fig.number, sub=(1,3,1), cmap="plasma", title=f"Truth ({freqs_mhz[fi]:.0f} MHz)")
healpy.mollview(resid_single,        fig=fig.number, sub=(1,3,2), cmap="bwr",   min=-vmax, max=vmax, title="Frac. residual: sc0")
healpy.mollview(resid_stacked,       fig=fig.number, sub=(1,3,3), cmap="bwr",   min=-vmax, max=vmax, title="Frac. residual: stacked")
plt.tight_layout()

rms_single  = float(np.std(resid_single[np.isfinite(resid_single)]))
rms_stacked = float(np.std(resid_stacked[np.isfinite(resid_stacked)]))
print(f"Sky frac. RMS — single: {rms_single*100:.2f}%   stacked: {rms_stacked*100:.2f}%")

In [ ]:
beam_truth  = campaign.beam.basis.deproject(campaign.beam.coeffs)
beam_single  = campaign.beam.basis.deproject(fit_single["params"]["beam_coeffs"])
beam_stacked = campaign.beam.basis.deproject(fit_stacked["params"]["beam_coeffs"])

fig = plt.figure(figsize=(14, 6))
for d in range(2):
    healpy.mollview(beam_truth[d, :, fi],   fig=fig.number, sub=(2,4,4*d+1), title=f"Dipole {d} truth")
    healpy.mollview(beam_single[d, :, fi],  fig=fig.number, sub=(2,4,4*d+2), title=f"Dipole {d} sc0")
    healpy.mollview(beam_stacked[d, :, fi], fig=fig.number, sub=(2,4,4*d+3), title=f"Dipole {d} stacked")
    healpy.mollview(
        beam_stacked[d, :, fi] - beam_single[d, :, fi],
        fig=fig.number, sub=(2,4,4*d+4), cmap="bwr", title=f"Dipole {d} stacked − sc0"
    )
plt.tight_layout()

for label, bc in [("sc0", fit_single["params"]["beam_coeffs"]),
                  ("stacked", fit_stacked["params"]["beam_coeffs"])]:
    rms = float(np.std(bc - campaign.beam.coeffs))
    print(f"Beam coeff RMS change ({label}): {rms:.6f}")

In [ ]:
snr_single  = float(np.linalg.norm(filtered_injected / sigma_noise.mean()))
snr_stacked = float(np.linalg.norm(filtered_injected / (sigma_noise.mean() / np.sqrt(2))))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(freqs_mhz, filtered_models.T, alpha=0.1, color="C0")
ax.plot(freqs_mhz, filtered_injected, "r-", lw=2, label="injected")
ax.set_xlabel("Frequency [MHz]"); ax.set_ylabel("Eigenmode-filtered signal [K]")
ax.set_title("21-cm Signal After GSM Eigenmode Filter")
ax.legend(); plt.tight_layout()

print(f"Combined SNR — single: {snr_single:.3f}   stacked (×√2): {snr_stacked:.3f}")

## 13. Pointing Knowledge And Noise Budget

In [ ]:
knowledge_deg = np.logspace(-3, 0, 20)
thermal_mK_single  = sigma_noise.mean() * 1e3
thermal_mK_stacked = thermal_mK_single / np.sqrt(2)  # twice the data
pointing_mK = 122.29 * knowledge_deg

plt.figure(figsize=(7, 3))
plt.loglog(knowledge_deg, np.hypot(thermal_mK_single,  pointing_mK), "C0-",  label="sc0 combined")
plt.loglog(knowledge_deg, np.hypot(thermal_mK_stacked, pointing_mK), "C1--", label="stacked combined")
plt.loglog(knowledge_deg, pointing_mK, "k:", label="pointing")
plt.axhline(thermal_mK_single,  color="C0", ls="-",  alpha=0.4, label=f"thermal sc0 ({thermal_mK_single:.1f} mK)")
plt.axhline(thermal_mK_stacked, color="C1", ls="--", alpha=0.4, label=f"thermal stacked ({thermal_mK_stacked:.1f} mK)")
plt.legend(fontsize=8); plt.xlabel("beam-orientation knowledge [deg]")
plt.ylabel("noise / leakage [mK]")
plt.tight_layout()

## Summary

- **`StackedForwardModel`** (section 9): wraps both spacecraft `ForwardModel` instances, concatenates outputs along the time axis, and is fully JAX-differentiable — no changes to `Calibrator` required.
- **Single-spacecraft calibration** (section 10): baseline joint sky+beam recovery from spacecraft-0 data.
- **Stacked calibration** (section 11): both spacecraft share sky and beam coefficients; doubled data volume and complementary sky coverage improve the constraint.
- **Comparison** (section 12): convergence curves, sky fractional residuals, beam map differences, and 21-cm eigenmode SNR for both cases.